In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_sessions.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_events.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-7_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-5_task-BreathCounting_electrodes.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-10_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-4_task-BreathCounting_eeg.json
/kaggle

In [2]:
"""Feature-family importance analysis for the 64-channel MWDataset.

This is the 64-EEG-channel adaptation of the supplied feature-wise notebook.
It extracts the same six families of features, trains the same grouped
cross-validated ExtraTrees model, and saves identically styled summary plots.
"""

import gc
import os
from pathlib import Path
import warnings

os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
os.environ.setdefault("MNE_NUM_JOBS", "1")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import butter, hilbert, sosfiltfilt, welch
from scipy.stats import kurtosis, skew
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold


# Kaggle paths for the attached MWDataset input. Environment variables allow
# the same file to be run locally without editing it.
DATASET_ROOT = Path(os.environ.get("MWDATASET_ROOT", "/kaggle/input/datasets/jvkrishwanth/mwdataset"))
OUTPUT_DIR = Path(os.environ.get("FEATURE_FAMILY_OUTPUT_DIR", "/kaggle/working/mwdataset_64ch_feature_family_results"))
SUBJECTS = ("sub-01", "sub-02")
SESSIONS = range(1, 12)
BDF_ARCHIVES = {
    "sub-01": {1: "sub-01-20260605T131512Z-3-002", 2: "sub-01-20260605T131512Z-3-003", 3: "sub-01-20260605T131512Z-3-002", 4: "sub-01-20260605T131512Z-3-002", 5: "sub-01-20260605T131512Z-3-002", 6: "sub-01-20260605T131512Z-3-001", 7: "sub-01-20260605T131512Z-3-001", 8: "sub-01-20260605T131512Z-3-001", 9: "sub-01-20260605T131512Z-3-001", 10: "sub-01-20260605T131512Z-3-003", 11: "sub-01-20260605T131512Z-3-003"},
    "sub-02": {1: "sub-02-20260605T131514Z-3-003", 2: "sub-02-20260605T131514Z-3-002", 3: "sub-02-20260605T131514Z-3-002", 4: "sub-02-20260605T131514Z-3-001", 5: "sub-02-20260605T131514Z-3-001", 6: "sub-02-20260605T131514Z-3-002", 7: "sub-02-20260605T131514Z-3-002", 8: "sub-02-20260605T131514Z-3-001", 9: "sub-02-20260605T131514Z-3-001", 10: "sub-02-20260605T131514Z-3-001", 11: "sub-02-20260605T131514Z-3-003"},
}
METADATA_ARCHIVES = {"sub-01": "sub-01-20260605T131512Z-3-001", "sub-02": "sub-02-20260605T131514Z-3-001"}
BANDS = {"Delta": (1.0, 4.0), "Theta": (4.0, 8.0), "Alpha": (8.0, 12.0), "Beta": (13.0, 30.0), "Gamma": (30.0, 45.0)}
TARGET_SFREQ = 256.0
EPOCH_SECONDS = 5.0
WINDOW_SECONDS, STEP_SECONDS = 1.5, 0.5
EVENT_TRIAL_START, EVENT_MW_REPORT, EVENT_START_COUNTING = 10, 30, 50
FOCUS_START_OFFSET, MW_START_OFFSET = 1.0, -5.0


def session_paths(subject, session):
    stem = f"{subject}_ses-{session}_task-BreathCounting"
    return (DATASET_ROOT / BDF_ARCHIVES[subject][session] / subject / "eeg" / f"{stem}_eeg.bdf",
            DATASET_ROOT / METADATA_ARCHIVES[subject] / subject / "eeg" / f"{stem}_channels.tsv")


def deduplicate_events(events):
    if not len(events):
        return events
    frame = pd.DataFrame(events, columns=["sample", "previous", "code"])
    return frame.drop_duplicates(["sample", "code"]).sort_values(["sample", "code"])[["sample", "previous", "code"]].to_numpy(dtype=int)


def make_epoch_events(events, sfreq, last_sample, epoch_samples):
    rows = []
    for sample, _, code in events:
        if code == EVENT_MW_REPORT:
            start, state = sample + round(MW_START_OFFSET * sfreq), 1
        elif code in (EVENT_TRIAL_START, EVENT_START_COUNTING):
            start, state = sample + round(FOCUS_START_OFFSET * sfreq), 0
        else:
            continue
        if 0 <= start and start + epoch_samples <= last_sample:
            rows.append([start, 0, state])
    return deduplicate_events(np.asarray(rows, dtype=int)) if rows else np.empty((0, 3), dtype=int)


def ica_clean(raw, eeg_names, exg_names, correlation_threshold=0.30):
    """Remove ICA components correlated with EXG channels without dropping trials."""
    cleaned = raw.copy()
    if not exg_names:
        return cleaned.pick(eeg_names)
    ica = mne.preprocessing.ICA(
        n_components=0.99, method="fastica", random_state=42, max_iter="auto",
    )
    ica.fit(cleaned, picks=eeg_names, verbose=False)
    scores = [
        np.asarray(ica.score_sources(cleaned, target=cleaned.get_data(picks=[name])[0]), dtype=float)
        for name in exg_names
    ]
    excluded = np.flatnonzero(np.max(np.abs(np.vstack(scores)), axis=0) >= correlation_threshold)
    ica.apply(cleaned, exclude=excluded, verbose=False)
    return cleaned.pick(eeg_names)


def bandpass(data, sfreq, low, high):
    sos = butter(4, [low / (sfreq / 2), high / (sfreq / 2)], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def shannon_entropy(x, bins=64):
    counts, _ = np.histogram(x, bins=bins)
    p = counts[counts > 0].astype(float)
    if not len(p):
        return 0.0
    p /= p.sum()
    return float(-np.sum(p * np.log2(p)))


def spectral_entropy(psd):
    p = np.asarray(psd, dtype=float)
    p = p[p > 0]
    if len(p) < 2:
        return 0.0
    p /= p.sum()
    return float(-np.sum(p * np.log2(p)) / np.log2(len(p)))


def differential_entropy_std(x, n_segments=6):
    values = [0.5 * np.log(2 * np.pi * np.e * max(np.var(part), 1e-20)) for part in np.array_split(x, n_segments)]
    return float(np.std(values))


def extended_channel_features(signal, psd, freqs, channel):
    signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)
    variance = np.var(signal)
    d1, d2 = np.diff(signal), np.diff(signal, n=2)
    mobility = np.sqrt(np.var(d1) / max(variance, 1e-20))
    complexity = np.sqrt(np.var(d2) / max(np.var(d1), 1e-20)) / max(mobility, 1e-20)
    whole, theta, alpha = ((freqs >= 1) & (freqs <= 45), (freqs >= 4) & (freqs < 8), (freqs >= 8) & (freqs < 12))
    band_psd, band_freqs = psd[whole], freqs[whole]
    return {
        f"{channel}_Mean": float(np.mean(signal)), f"{channel}_RMS": float(np.sqrt(np.mean(signal ** 2))),
        f"{channel}_Variance": float(variance), f"{channel}_Energy": float(np.sum(signal ** 2)),
        f"{channel}_Peak2Peak": float(np.ptp(signal)), f"{channel}_Kurtosis": float(kurtosis(signal, fisher=True, bias=False)),
        f"{channel}_Skewness": float(skew(signal, bias=False)), f"{channel}_HjorthActivity": float(variance),
        f"{channel}_HjorthMobility": float(mobility), f"{channel}_HjorthComplexity": float(complexity),
        f"{channel}_ShannonEntropy": shannon_entropy(signal), f"{channel}_SpectralEntropy": spectral_entropy(band_psd),
        f"{channel}_DifferentialEntropyStd": differential_entropy_std(signal),
        f"{channel}_TotalPower": float(np.trapezoid(band_psd, band_freqs)) if len(band_psd) else 0.0,
        f"{channel}_DominantFreq": float(band_freqs[np.argmax(band_psd)]) if len(band_psd) else 0.0,
        f"{channel}_ThetaPower": float(np.mean(psd[theta])) if np.any(theta) else 0.0,
        f"{channel}_AlphaPower": float(np.mean(psd[alpha])) if np.any(alpha) else 0.0,
    }


def window_features(data, sfreq, channels, subject, session, epoch_index, window_index, label):
    freqs, psd = welch(data, fs=sfreq, nperseg=min(256, data.shape[-1]), axis=-1)
    envelopes = {band: np.abs(hilbert(bandpass(data, sfreq, *BANDS[band]), axis=-1)) for band in ("Alpha", "Theta")}
    alpha_corr = np.nan_to_num(np.corrcoef(envelopes["Alpha"]), nan=0.0)
    phases = {band: np.angle(hilbert(bandpass(data, sfreq, *BANDS[band]), axis=-1)) for band in ("Alpha", "Theta")}
    # Matrix multiplication evaluates every channel-pair PLV at once; this is
    # equivalent to the pairwise reference calculation but crucial for 64 EEG
    # channels (2,016 pairs per band).
    plv = {}
    for band, phase in phases.items():
        unit_phase = np.exp(1j * phase)
        plv[band] = np.abs(unit_phase @ unit_phase.conj().T / phase.shape[-1])
    row = {"Subject": subject, "Session": session, "Epoch_Index": epoch_index, "Window_Index": window_index, "Label": label}
    total = (freqs >= 1) & (freqs <= 45)
    for i, channel in enumerate(channels):
        channel_psd = psd[i]
        total_power = np.mean(channel_psd[total]) if np.any(total) else 1e-20
        for band, (low, high) in BANDS.items():
            mask = (freqs >= low) & (freqs < high if band != "Gamma" else freqs <= high)
            absolute = np.mean(channel_psd[mask]) if np.any(mask) else 0.0
            row[f"{channel}_{band}_Abs"] = absolute
            row[f"{channel}_{band}_Rel"] = absolute / max(total_power, 1e-20)
        for band in ("Alpha", "Theta"):
            env = envelopes[band][i]; mean = np.mean(env); threshold = mean + 2 * np.std(env)
            row.update({f"{channel}_{band}_Mean": float(mean), f"{channel}_{band}_Var": float(np.var(env)), f"{channel}_{band}_CV": float(np.std(env) / max(mean, 1e-20)), f"{channel}_{band}_Bursts": float(np.sum(np.diff((env > threshold).astype(int)) == 1))})
        row[f"{channel}_Envelope_Sync"] = float(np.mean(alpha_corr[i, np.arange(len(channels)) != i]))
        row.update(extended_channel_features(data[i], channel_psd, freqs, channel))
    for band in ("Alpha", "Theta"):
        for i in range(len(channels)):
            for j in range(i + 1, len(channels)):
                row[f"PLV_{band}_{channels[i]}_{channels[j]}"] = float(plv[band][i, j])
    return row


def extract_dataset():
    rows = []
    for subject in SUBJECTS:
        for session in SESSIONS:
            bdf_path, channels_path = session_paths(subject, session)
            print(f"Reading {subject}, session {session}", flush=True)
            raw = mne.io.read_raw_bdf(bdf_path, preload=False, stim_channel="Status", verbose=False)
            events = mne.find_events(raw, stim_channel="Status", shortest_event=1, consecutive=True, verbose=False)
            table = pd.read_csv(channels_path, sep="\t")
            channels = [name for name in table.loc[table["channelTypes"].fillna("").str.upper().eq("EEG"), "name"].astype(str) if name in raw.ch_names]
            if len(channels) != 64:
                raise RuntimeError(f"{subject}, session {session}: expected 64 EEG channels, found {len(channels)}.")
            original_sfreq = raw.info["sfreq"]
            # Preload each session once. Repeated BDF reads per short epoch are far
            # slower than one bounded per-session load.
            exg_names = [f"EXG{i}" for i in range(1, 9) if f"EXG{i}" in raw.ch_names]
            raw.pick(channels + exg_names).load_data()
            raw.resample(TARGET_SFREQ, npad="auto", verbose=False)
            events[:, 0] = np.rint(events[:, 0] * TARGET_SFREQ / original_sfreq).astype(int)
            raw.filter(0.5, 45.0, fir_design="firwin", verbose=False)
            raw = ica_clean(raw, channels, exg_names)
            epoch_samples = round(EPOCH_SECONDS * TARGET_SFREQ)
            epoch_events = make_epoch_events(deduplicate_events(events), TARGET_SFREQ, raw.n_times - 1, epoch_samples)
            window_n, step_n = round(WINDOW_SECONDS * TARGET_SFREQ), round(STEP_SECONDS * TARGET_SFREQ)
            for epoch_index, event in enumerate(epoch_events):
                label = event[2]
                n_epoch = round(5.0 * TARGET_SFREQ) if label == 0 else round(4.9 * TARGET_SFREQ)
                epoch = raw.get_data(start=event[0], stop=event[0] + n_epoch)
                for window_index, start in enumerate(range(0, epoch.shape[-1] - window_n + 1, step_n)):
                    rows.append(window_features(epoch[:, start:start + window_n], TARGET_SFREQ, channels, subject, session, epoch_index, window_index, int(event[2])))
            print(f"  Extracted {len(rows)} cumulative windows", flush=True)
            del raw; gc.collect()
    return pd.DataFrame(rows)


def feature_family(name):
    f = name.lower()
    if f.startswith("plv_") or "envelope_sync" in f: return "Functional connectivity / synchrony"
    if "hjorth" in f: return "Hjorth features"
    if any(x in f for x in ("spectralentropy", "shannonentropy", "differentialentropy", "sampleentropy", "fractal", "hurst")): return "Entropy and complexity"
    if any(x in f for x in ("_alpha_mean", "_alpha_var", "_alpha_cv", "_alpha_bursts", "_theta_mean", "_theta_var", "_theta_cv", "_theta_bursts")): return "Oscillatory-envelope dynamics"
    if any(x in f for x in ("_rms", "_mean", "_variance", "_energy", "_peak2peak", "_kurtosis", "_skewness")): return "Time-domain statistics"
    if "totalpower" in f or "dominantfreq" in f: return "Spectral summary features"
    return "Band-power features"


def analyse(df):
    meta = ["Subject", "Session", "Epoch_Index", "Window_Index", "Label"]
    x, y, groups = df.drop(columns=meta).apply(pd.to_numeric, errors="coerce"), df.Label.to_numpy(), df.Subject.to_numpy()
    duplicates = x.T.duplicated(); print(f"Removing {duplicates.sum()} exact duplicate feature columns.", flush=True)
    x = x.loc[:, ~duplicates]; feature_names = x.columns.to_numpy()
    cv = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)
    importances = []
    for fold, (train, test) in enumerate(cv.split(x, y, groups), 1):
        imputer = SimpleImputer(strategy="median"); x_train = imputer.fit_transform(x.iloc[train]); x_test = imputer.transform(x.iloc[test])
        selector = SelectKBest(f_classif, k=min(100, x_train.shape[1])); x_train = selector.fit_transform(x_train, y[train]); x_test = selector.transform(x_test)
        selected = feature_names[selector.get_support()]
        model = ExtraTreesClassifier(n_estimators=500, class_weight="balanced", max_features="sqrt", random_state=fold, n_jobs=-1)
        model.fit(x_train, y[train])
        fold_importance = pd.Series(0.0, index=feature_names); fold_importance.loc[selected] = model.feature_importances_; importances.append(fold_importance)
    summary = pd.concat(importances, axis=1).mean(axis=1).rename("Importance").reset_index().rename(columns={"index": "Feature"})
    summary["Feature family"] = summary.Feature.map(feature_family); summary = summary.sort_values("Importance", ascending=False)
    family = summary.groupby("Feature family", as_index=False).agg(Feature_count=("Feature", "count"), Total_importance=("Importance", "sum"), Mean_importance_per_feature=("Importance", "mean")).sort_values("Mean_importance_per_feature", ascending=False)
    summary.to_csv(OUTPUT_DIR / "feature_importance_by_feature.csv", index=False); family.to_csv(OUTPUT_DIR / "feature_importance_by_umbrella.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    sns.barplot(data=family, x="Mean_importance_per_feature", y="Feature family", hue="Feature family", legend=False, palette="viridis", ax=axes[0]); axes[0].set_title("Usefulness per Feature (fair comparison)"); axes[0].set_xlabel("Mean cross-validated importance")
    sns.barplot(data=family.sort_values("Total_importance", ascending=False), x="Total_importance", y="Feature family", hue="Feature family", legend=False, palette="magma", ax=axes[1]); axes[1].set_title("Total Contribution by Feature Family"); axes[1].set_xlabel("Summed cross-validated importance")
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / "feature_family_comparison.png", dpi=250); plt.close()
    plt.figure(figsize=(12, 9)); sns.barplot(data=summary.head(25), x="Importance", y="Feature", hue="Feature family", dodge=False, palette="tab10"); plt.title("Top Features: Mind Wandering vs Focus"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.savefig(OUTPUT_DIR / "top_individual_features.png", dpi=250); plt.close()
    print(f"Saved results: {OUTPUT_DIR}")


def main():
    warnings.filterwarnings("ignore"); mne.set_log_level("WARNING"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    dataframe = extract_dataset(); dataframe.to_csv(OUTPUT_DIR / "windowed_features_all_families.csv", index=False); analyse(dataframe)


if __name__ == "__main__":
    main()


Reading sub-01, session 1
  Extracted 422 cumulative windows
Reading sub-01, session 2
  Extracted 739 cumulative windows
Reading sub-01, session 3
  Extracted 1186 cumulative windows
Reading sub-01, session 4
  Extracted 1563 cumulative windows
Reading sub-01, session 5
  Extracted 1984 cumulative windows
Reading sub-01, session 6
  Extracted 2542 cumulative windows
Reading sub-01, session 7
  Extracted 2957 cumulative windows
Reading sub-01, session 8
  Extracted 3311 cumulative windows
Reading sub-01, session 9
  Extracted 3732 cumulative windows
Reading sub-01, session 10
  Extracted 4207 cumulative windows
Reading sub-01, session 11
  Extracted 4794 cumulative windows
Reading sub-02, session 1
  Extracted 5089 cumulative windows
Reading sub-02, session 2
  Extracted 5302 cumulative windows
Reading sub-02, session 3
  Extracted 5560 cumulative windows
Reading sub-02, session 4
  Extracted 5803 cumulative windows
Reading sub-02, session 5
  Extracted 6075 cumulative windows
Reading 